# LanceDB - a vector database for LLM applications

In [36]:
import lancedb

db = lancedb.connect(uri = "vector_database")
db

LanceDBConnection(uri='c:\\Users\\felix\\Documents\\github\\AI_ops_DE25\\course_code\\09_lacedb\\vector_database')

In [37]:
db.uri

'c:\\Users\\felix\\Documents\\github\\AI_ops_DE25\\course_code\\09_lacedb\\vector_database'

## Create a table

In [38]:
import json

with open("animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [39]:
table_animals = db.create_table("animals_text", exist_ok=True, data=data)
table_animals

LanceTable(name='animals_text', _conn=LanceDBConnection(uri='c:\\Users\\felix\\Documents\\github\\AI_ops_DE25\\course_code\\09_lacedb\\vector_database'))

In [40]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [41]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

table_animals.add(more_data)

AddResult(version=5)

In [42]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create empty table then delete it

In [43]:
# LanceModel is a Pydantic model that can be converted to LanceDB table
from lancedb.pydantic import LanceModel, Vector

class JokeSchema(LanceModel):
    joke: str 
    rating: int


db.create_table(name = "jokes",schema=JokeSchema, exist_ok=True)

LanceTable(name='jokes', _conn=LanceDBConnection(uri='c:\\Users\\felix\\Documents\\github\\AI_ops_DE25\\course_code\\09_lacedb\\vector_database'))

In [44]:
db.table_names()

C:\Users\felix\AppData\Local\Temp\ipykernel_16540\763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text', 'jokes']

In [45]:
db.drop_table("jokes")

In [46]:
db.table_names()

C:\Users\felix\AppData\Local\Temp\ipykernel_16540\763631344.py:1: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  db.table_names()


['animals_text']

## Open existing table

In [47]:
db.open_table("animals_text").head(2)

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1]]]

## Vector search in LanceDB

In [48]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [49]:
query_vector = [0.5, 0.2, 0.9]

table_animals.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354


## Embedding API

- idea: want to put in a text -> and it will automagically generate vector embeddings
- put in a query and it will automagically generate vector embeddings
- calculate closest distances

In [50]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from lancedb.embeddings import TextEmbeddingFunction, register, get_registry

load_dotenv()


@register("nvidia-embed")
class NvidiaEmbeddings(TextEmbeddingFunction):
    name: str = "nvidia/nemotron-3.5-embed-1b"

    def ndims(self):
        return 2048

    def _client(self):
        return OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=os.environ["NVIDIA_API"],
        )

    def _embed(self, texts, input_type):
        if isinstance(texts, str):
            texts = [texts]
        else:
            texts = list(texts)

        resp = self._client().embeddings.create(
            model=self.name,
            input=texts,
            extra_body={"input_type": input_type},
        )
        return [d.embedding for d in resp.data]

    def generate_embeddings(self, texts):
        return self._embed(texts, "passage")

    def compute_query_embeddings(self, query, *args, **kwargs):
        return self._embed(query, "query")

KeyError: 'nvidia-embed was already registered'

In [51]:
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API"],
)

[m.id for m in client.models.list() if "embed" in m.id]

['nvidia/embed-qa-4',
 'nvidia/llama-3.2-nemoretriever-1b-vlm-embed-v1',
 'nvidia/llama-3.2-nv-embedqa-1b-v1',
 'nvidia/llama-nemotron-embed-vl-1b-v2',
 'nvidia/nemotron-3-embed-1b',
 'nvidia/nemotron-3.5-embed-1b',
 'nvidia/nemotron-3.5-embed-8b',
 'nvidia/nv-embedqa-mistral-7b-v2',
 'snowflake/arctic-embed-l']

In [52]:
resp = client.embeddings.create(
    model="nvidia/nemotron-3.5-embed-1b",
    input=["hello"],
    extra_body={"input_type": "query"},
)
len(resp.data[0].embedding)

2048

In [53]:
def ndims(self):
    return 2048

In [54]:
embedder = get_registry().get("nvidia-embed").create()
import numpy as np

hello_embedding = np.array(embedder.generate_embeddings("hello"))

hello_embedding.shape

(1, 2048)

In [55]:
class JokeModel(LanceModel):
    joke: str = embedder.SourceField()
    vector: Vector(2048) = embedder.VectorField()

table_jokes = db.create_table("jokes", schema=JokeModel, exist_ok=True)

In [56]:
import pandas as pd

with open("jokes.json", "r", encoding="utf-8") as file:
    jokes_data = json.loads(file.read())

df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [57]:
df_jokes["joke"].iloc[0]

'Parallel lines have so much in common—it’s sad they’ll never meet.'

## Add data to table

In [ ]:
table_jokes.add(df_jokes)